# Skeleton for using jupyter on UTE Cloud Testlines

The below code is used, to determine the current testline configuration by the host name.

With the host name information, the testline configuration is loaded, so you can later on
work with the common keywords on the testline.

You need to execute the cell (by pressing [Ctrl]+[Enter] on the cell or pressing the play button)

After running the cell, an ip address is printed. If the ip address corresponds to the address of the local machine,
the correct testline configuration is loaded.

**You might need to update the location of the testline_configuration repository**

In [ ]:
*** Settings ***
Library    jupyter_helper.py
Resource    ../ute_workarounds.robot

*** Test Cases ***
Load TL Config
    Load Local Testline Config       /home/ute/testline_configuration
    Set Robotws Path                 /home/ute/DEVELOPMENT/robotws
    UTE Cloud Workaround
    Log    HostPC is ${tl.test_pc}
    ${host_ip}=    Get Host IP
    ${local_ip}=   Set Variable   ${tl.test_pc.default_ssh_connection_details.addr}
    Should Be Equal    ${host_ip}   ${local_ip}    The tl config IP ${local_ip} does not fit the host ip ${host_ip}
    Log To Console    "Hello World!"
    Set Variable   ${local_ip}
    

After having the testline configuration loaded with the first cell on this page, you can start interacting with your testline. 

The next cell is using the keywords from ute_workarounds to bring the testline in the OnAir status

After executing this Keyword, you are able to proceed with further interactions, like attaching UES, collecting logs, ...

In [ ]:
*** Settings ***
Resource    ../ute_workarounds.robot

*** Test Cases ***
Bring Airphone to on Air status
    UTE Cloud Workaround
    ${wts_status}=      Run Keyword and Return Status      Bring Airphone TL to on Air status without reboot
    Run Keyword If      ${wts_status} == False             Bring Airphone TL to on Air status

# Test Setup and Teardown to connect to running egate and edaemon

Since Jupyter is instanciating the ute_wtssim again again, with each start of a new cell, the alias from a previous run cell is no more valid. A workaround for this is to define a test setup and a test teardown to setup the utw_wtssim with each test case, and tear it down after each test case.

**Don't run the keywords directly, this will lead to errors.**

In [ ]:
*** Settings ***
Library     ute_wtssim
Resource    ../ute_workarounds.robot
Test Setup      Connect to running egate
Test Teardown   Teardown the wts sim
    

*** Keywords ***
Connect to running egate
    ${wts_alias}=      Set Variable    ap_egate
    ${egate_port}=     Set Variable    ${tl.core.egate_port}
    ${edaemon_alias}=  Set Variable    ap_edaemon
    ${edaemon_port}=   Set Variable    ${tl.core.edaemon_port}
    Set Suite Variable    ${wts_alias}
    Set Suite Variable    ${edaemon_alias}

    ute_wtssim.setup_egate    egate_port=${egate_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${wts_alias}
    ute_wtssim.setup_edaemon    edaemon_port=${edaemon_port}  host_ip=${tl.core.default_ssh_connection_details.addr}
    ...    tool_dir=${tl.core.dir_path}  alias=${edaemon_alias}
    ${wts_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_info    pass_regexp=Process ID    wts_alias=${wts_alias}
    ${sim_status}=    Run Keyword and Return Status
    ...    Send Egate Command and verify output    cmd=sim_state_disp    pass_regexp=The state is ACTIVE    wts_alias=${wts_alias}
    
    Run Keyword If    $wts_status == False or $sim_status == False    Fail   Unable To Connect to running Egate

Teardown the wts sim
    ute_wtssim.teardown_egate     alias=${wts_alias}
    ute_wtssim.teardown_edaemon   alias=${edaemon_alias}


In [ ]:
%%python module JupyterDisplay

from IPython.display import display_pretty, Markdown, display, Code, JSON

def display_markdown(text):
    display(Markdown(text))
    
def display_json(output_object):
    display(JSON(output_object))

def display_text(text):
    display(Code(text.decode('utf-8')))

In [ ]:
*** Settings ***
Library    JupyterDisplay

*** Test Cases ***
Example Test Case
    Display    Hello World!

In [ ]:
%%python module InstallWiresharkPlugins
import subprocess
import os
import logging
from urllib.parse import urlparse

import sys
sys.path.append('/home/ute/DEVELOPMENT/robotws')
from testsuite.Ulm.resources.Common.python.pmcounters import python_log


from IPython.display import display_pretty, Markdown, display, Code, JSON

logger = logging.getLogger()

@python_log
def is_install_script_existing(directory='/tmp'):
    return 'install_wslte_plugins.bash' in os.listdir(directory)

@python_log
def remove_install_script(directory='/tmp'):
    logger.info('Install script exists: {}'.format(is_install_script_existing()))
    os.remove(os.path.join(directory, 'install_wslte_plugins.bash'))
    logger.info('Install script exists: {}'.format(is_install_script_existing()))
    

@python_log
def get_available_plugins():
    stdout, stderr = subprocess.Popen(['tshark', '-G', 'plugins'], stdout=subprocess.PIPE, stderr=subprocess.PIPE).communicate()
    logger.info('STDERR: ' + stderr.decode('utf-8'))
    return stdout.decode('utf-8')

@python_log    
def show_available_plugins():
    display(Code(get_available_plugins()))
    logger.info(get_available_plugins)

@python_log
def download_install_script(directory='/tmp', force=True):
    if force and is_install_script_existing(directory):
        remove_install_script(directory)
            
    script_url = 'http://artifactory-hz1.int.net.nokia.com/artifactory/wslte5g_plugins-local/tools/install_wslte_plugins.bash'
    stdout, stderr = subprocess.Popen(['wget', '-v', script_url], stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=directory).communicate()

    logger.info(stdout.decode('utf-8'))
    logger.info(stderr.decode('utf-8'))

@python_log
def run_install_script_for_tshark_plugins(gnb_sw_version, directory='/tmp'):
    install_script_name = os.path.join(directory, 'install_wslte_plugins.bash')
    subprocess.Popen(['chmod', '+x', install_script_name]).communicate()
    cmd = [install_script_name, '--sbts', gnb_sw_version]
    stdout, stderr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE).communicate()
    logger.info('STDOUT:' + stdout.decode('utf-8'))
    logger.info('STDERR:' + stderr.decode('utf-8'))
    #display(Code(stdout.decode('utf-8')))

@python_log    
def uninstall_plugin():
    plugin_table = get_available_plugins()
    mcshark_plugins = [x for x in plugin_table.split('\n') if x and 'mcshark' in x]
    
    mcshark_plugins = [dict(zip(('library', 'version', 'plugin_type', 'plugin_path'), bb.split())) for bb in mcshark_plugins]
    
    mcshark_plugin_paths = [x['plugin_path'] for x in mcshark_plugins]
    
    for plugin_path in mcshark_plugin_paths:
        logger.info('Removing [{}]'.format(plugin_path))
        os.remove(plugin_path)
    
    plugin_path = '/home/ute/.local/lib/wireshark/plugins/3.4/epan'
    left_plugins = [os.path.join(plugin_path, x) for x in os.listdir(plugin_path) if 'mcshark' in x]
    for left_file in left_plugins:
        os.remove(left_file)
    
    

@python_log
def download_mcshark_for_airphone(directory='/tmp'):
    ap_plugin_url = 'https://artifactory-hz1.int.net.nokia.com/artifactory/wslte5g_plugins-local/MCShark_AirPhone/4.0.20132/mcshark-3.4-1.15.0-4.0.20132_l1_REL_14_UE+buster.so'
    
    ap_plugin_url_components = urlparse(ap_plugin_url)
    _, downloaded_file_name = os.path.split(ap_plugin_url_components.path)
    target_filename = os.path.join(directory, downloaded_file_name)
    if os.path.exists(target_filename):
        os.remove(target_filename)
    
    stdout, stderr = subprocess.Popen(['wget', '-v', ap_plugin_url], stdout=subprocess.PIPE, stderr=subprocess.PIPE, cwd=directory).communicate()
    logger.info(stdout.decode('utf-8'))
    
    output = stderr.decode('utf-8')
    
    logger.info(output)
    
    saving_to = [x for x in output.split('\n') if x.strip().startswith('Saving to')]
    if len(saving_to) == 1:
        # Saving to: ‘mcshark-3.4-1.15.0-4.0.20132_l1_REL_14_UE+buster.so’
        _, filename = saving_to[0].split(':')
        filename = filename.strip()[1:-1]
        return os.path.join(directory, filename)

@python_log
def install_additional_mcshark_dissector(filename):
    
    plugin_path = '/home/ute/.local/lib/wireshark/plugins/3.4/epan'
    stdout, stderr = subprocess.Popen(['cp', '-v', filename, plugin_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE).communicate()

    logger.info(stdout.decode('utf-8'))
    logger.info(stderr.decode('utf-8'))    
    
    
    

    

In [ ]:
*** Settings ***
Resource    ../testline.robot
Library     InstallWiresharkPlugins

*** Test Cases ***
Check the SW Version
    ${gnb_sw_version}    Get gNB SW Version
    Display    ${gnb_sw_version}
    
    Is Install Script Existing
    
    Show Available Plugins
    
    Download Install Script
    
    Run Install Script for TShark Plugins   ${gnb_sw_version}
    
    Show Available Plugins
    
    Uninstall Plugin
    
    Show Available Plugins
    
    ${ap_dissector}=    Download MCShark For Airphone
    Install Additional Mcshark Dissector   ${ap_dissector}
    
    Show Available Plugins
    #Uninstall Plugin